# 02 — Preprocessing Pipeline
**exoplanet-ml-classifier** | MLEA_M — ECI 2026-1

This notebook walks through every preprocessing decision made before model training:
leakage removal, feature/target split, class imbalance handling, and the
train/val/test split.

---

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('ignore')

from src.constants import (
    NUMERIC_FEATURES,
    PROCESSED_DATA_PATH,
    RAW_DATA_PATH,
    RANDOM_SEED,
    TARGET_COLUMN,
    TEST_SIZE,
    VALIDATION_SIZE,
)
from src.data_loader import (
    drop_leakage_columns,
    get_feature_target_split,
    load_raw_koi_data,
)
from src.preprocessing import apply_smote_if_imbalanced, split_data

## Step 1 — Load Raw Data

In [2]:
df_raw = load_raw_koi_data(ROOT / RAW_DATA_PATH)
print(f'Shape before leakage removal: {df_raw.shape}')

Loaded dataset — shape: (9564, 49)
First 3 rows:
      kepid kepoi_name   kepler_name koi_disposition koi_pdisposition  koi_score  koi_fpflag_nt  koi_fpflag_ss  koi_fpflag_co  koi_fpflag_ec  koi_period  koi_period_err1  koi_period_err2  koi_time0bk  koi_time0bk_err1  koi_time0bk_err2  koi_impact  koi_impact_err1  koi_impact_err2  koi_duration  koi_duration_err1  koi_duration_err2  koi_depth  koi_depth_err1  koi_depth_err2  koi_prad  koi_prad_err1  koi_prad_err2  koi_teq  koi_teq_err1  koi_teq_err2  koi_insol  koi_insol_err1  koi_insol_err2  koi_model_snr  koi_tce_plnt_num koi_tce_delivname  koi_steff  koi_steff_err1  koi_steff_err2  koi_slogg  koi_slogg_err1  koi_slogg_err2  koi_srad  koi_srad_err1  koi_srad_err2         ra        dec  koi_kepmag
0  10797460  K00752.01  Kepler-227 b       CONFIRMED        CANDIDATE      1.000              0              0              0              0    9.488036         0.000028        -0.000028   170.538750          0.002160         -0.002160       0

## Step 2 — Drop Leakage Columns

Several columns encode the label directly (`koi_disposition`, `koi_score`) or are
post-hoc flags computed after the classification was already made (`koi_fpflag_*`).
Including them would cause **data leakage**, inflated training metrics that would not
generalise to new candidates.

In [3]:
df_clean = drop_leakage_columns(df_raw)
print(f'Shape after leakage removal : {df_clean.shape}')
print(f'Columns removed             : {df_raw.shape[1] - df_clean.shape[1]}')

Dropping 9 leakage columns: ['koi_score', 'koi_disposition', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'kepid', 'kepoi_name', 'kepler_name']
Shape after leakage removal : (9564, 40)
Columns removed             : 9


## Step 3 — Feature / Target Split

In [4]:
X, y = get_feature_target_split(df_clean)

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'y dtype : {y.dtype}')
print(f'Class distribution:\n{y.value_counts().to_frame("count")}')

X shape : (9564, 39)
y shape : (9564,)
y dtype : int64
Class distribution:
                  count
koi_pdisposition       
0                  4847
1                  4717


## Step 4 — Class Imbalance Check → Apply SMOTE If Needed

SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples in the
feature space of the minority class by interpolating between existing minority-class
instances and their k-nearest neighbours.  It is only applied when the minority/majority
ratio falls below the configured threshold (0.3 by default).

In [5]:
from sklearn.impute import SimpleImputer

available = [c for c in NUMERIC_FEATURES if c in X.columns]
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X[available])

X_resampled, y_resampled = apply_smote_if_imbalanced(
    X_imputed, y.values, threshold=0.3, random_state=RANDOM_SEED
)

print(f'Post-SMOTE X shape : {X_resampled.shape}')
unique, counts = np.unique(y_resampled, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f'  Class {cls}: {cnt}')

Class ratio 0.973 >= threshold 0.3 — SMOTE not applied.
Post-SMOTE X shape : (9564, 13)
  Class 0: 4847
  Class 1: 4717


## Step 5 — Train / Val / Test Split

The data is split into three stratified partitions:

| Partition  | Fraction |
|------------|----------|
| **Test**   | 20%      |
| **Val**    | 10% of remaining (≈8% total) |
| **Train**  | ≈72% total |

Stratified splits ensure the positive class proportion is preserved across all three sets.

In [6]:
X_df = pd.DataFrame(X_resampled, columns=available)
y_s = pd.Series(y_resampled, name=TARGET_COLUMN)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X_df, y_s, TEST_SIZE, VALIDATION_SIZE, RANDOM_SEED
)

splits = {
    'train': (X_train, y_train),
    'val'  : (X_val,   y_val),
    'test' : (X_test,  y_test),
}

print('Stratified distribution check:')
for split_name, (X_s, y_split) in splits.items():
    pos = y_split.mean()
    print(f'  {split_name:5s}  n={len(y_split):5d}  positive_rate={pos:.3f}')

Split sizes — train: 6885 (72.0%), val: 766 (8.0%), test: 1913 (20.0%)
  train positive rate: 0.493
  val positive rate: 0.493
  test positive rate: 0.493
Stratified distribution check:
  train  n= 6885  positive_rate=0.493
  val    n=  766  positive_rate=0.493
  test   n= 1913  positive_rate=0.493


## Step 6 — Preprocessor Explanation

The `build_preprocessor()` function returns a `ColumnTransformer` that applies the
following two-step pipeline to every column in `NUMERIC_FEATURES`:

1. **Median imputation** — replaces NaN with the column median, which is robust to the
   right-skewed distributions observed in planetary/stellar parameter columns.

2. **Z-score standardisation** (StandardScaler) — transforms each feature to have
   zero mean and unit variance:

$$z = \frac{x - \mu}{\sigma}$$

where $\mu$ is the training-set mean and $\sigma$ is the training-set standard deviation.
Standardisation is required for distance-based models (KNN), regularised linear models
(Logistic Regression), and gradient-based optimisers (MLP). Tree-based models (Random
Forest, XGBoost) are scale-invariant but are standardised here for API consistency.

## Step 7 — Save Processed Splits to Disk

In [7]:
processed_dir = ROOT / 'data/processed'
processed_dir.mkdir(parents=True, exist_ok=True)

for split_name, (X_split, y_split) in splits.items():
    out_path = processed_dir / f'koi_{split_name}.csv'
    combined = X_split.copy()
    combined[TARGET_COLUMN] = y_split.values
    combined.to_csv(out_path, index=False)
    print(f'Saved {split_name} split to {out_path}')

print('All splits saved successfully.')

Saved train split to C:\Users\Usuario\exoplanet-ml-classifier\data\processed\koi_train.csv
Saved val split to C:\Users\Usuario\exoplanet-ml-classifier\data\processed\koi_val.csv
Saved test split to C:\Users\Usuario\exoplanet-ml-classifier\data\processed\koi_test.csv
All splits saved successfully.
